<a href="https://colab.research.google.com/github/Padmadharshini03/Machine-learning-/blob/main/experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import io

data_str = """Sky,AirTemp,Humidity,Wind,Water,Forecast,EnjoySport
Sunny,Warm,Normal,Strong,Warm,Same,Yes
Sunny,Warm,High,Strong,Warm,Same,Yes
Rainy,Cold,High,Strong,Warm,Change,No
Sunny,Warm,High,Strong,Cool,Change,Yes"""

# Save the data as enjoysport.csv
with open('enjoysport.csv', 'w') as f:
    f.write(data_str)

print("enjoysport.csv successfully created!")


enjoysport.csv successfully created!


In [2]:
# Program 2: Candidate-Elimination algorithm
import pandas as pd

data = pd.read_csv('enjoysport.csv')
print("Training data:")
print(data)
print()

concepts = data.iloc[:, :-1].values
target = data.iloc[:, -1].values
n = concepts.shape[1]

# Possible values of every attribute (needed to specialise G)
domains = [sorted(set(concepts[:, j])) for j in range(n)]


def covers(h, x):
    """True if hypothesis h classifies instance x as positive."""
    return all(hj == '?' or hj == xj for hj, xj in zip(h, x))


def more_general_or_equal(h1, h2):
    """True if h1 is more general than (or equal to) h2."""
    return all(a == '?' or a == b or b == '0' for a, b in zip(h1, h2))


# Initial boundaries
S = ['0'] * n            # most specific boundary
G = [['?'] * n]          # most general boundary

print("Initial S:", S)
print("Initial G:", G)
print()

for i, x in enumerate(concepts):
    if str(target[i]).strip().lower() == 'yes':
        # Positive example: remove inconsistent members of G, generalise S
        G = [g for g in G if covers(g, x)]
        for j in range(n):
            if S[j] == '0':
                S[j] = x[j]
            elif S[j] != x[j]:
                S[j] = '?'
    else:
        # Negative example: specialise members of G that cover x
        new_G = []
        for g in G:
            if not covers(g, x):
                new_G.append(g)
                continue
            for j in range(n):
                if g[j] == '?':
                    for value in domains[j]:
                        if value != x[j]:
                            h = list(g)
                            h[j] = value
                            if more_general_or_equal(h, S):
                                new_G.append(h)
        # keep only the maximally general members, remove duplicates
        pruned = []
        for h in new_G:
            if h in pruned:
                continue
            if any(o != h and more_general_or_equal(o, h) for o in new_G):
                continue
            pruned.append(h)
        G = pruned

    print(f"Example {i + 1}: {list(x)} -> {target[i]}")
    print("   S =", S)
    print("   G =", G)
    print()

print("Final Specific boundary S:")
print(S)
print("Final General boundary G:")
for g in G:
    print(g)


Training data:
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High  Strong  Warm   Change         No
3  Sunny    Warm     High  Strong  Cool   Change        Yes

Initial S: ['0', '0', '0', '0', '0', '0']
Initial G: [['?', '?', '?', '?', '?', '?']]

Example 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] -> Yes
   S = ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
   G = [['?', '?', '?', '?', '?', '?']]

Example 2: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] -> Yes
   S = ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
   G = [['?', '?', '?', '?', '?', '?']]

Example 3: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] -> No
   S = ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
   G = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']]

Example 4: ['S